# Credit Card Fraud Detector


End-to-end pipeline: 284,807 transactions, 0.17% fraud, Target ROC-AUC ≥ 0.95

## Setup — install libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded successfully')

---
## Phase 1 — Data loading & audit

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
print('\nMissing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())

In [ ]:
class_counts = df['Class'].value_counts()
fraud_pct = df['Class'].mean() * 100
print(f'Legitimate transactions: {class_counts[0]:,}')
print(f'Fraud transactions:      {class_counts[1]:,}')
print(f'Fraud rate:              {fraud_pct:.4f}%')

---
## Phase 2 — EDA
Find patterns that separate fraud from legitimate transactions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution
for cls, label, color in [(0,'Legitimate','#2196F3'),(1,'Fraud','#F44336')]:
    axes[0].hist(df[df['Class']==cls]['Amount'], bins=60,
                 alpha=0.6, label=label, color=color)
axes[0].set_title('Transaction amount by class')
axes[0].set_xlabel('Amount (USD)')
axes[0].set_yscale('log')
axes[0].legend()

# Time distribution
for cls, label, color in [(0,'Legitimate','#2196F3'),(1,'Fraud','#F44336')]:
    axes[1].hist(df[df['Class']==cls]['Time']/3600, bins=48,
                 alpha=0.6, label=label, color=color)
axes[1].set_title('Transaction time by class (hours)')
axes[1].set_xlabel('Hours since first transaction')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/eda_amount_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Fraud peaks at low-traffic hours — suggests automated card testing')

In [ ]:
# Top discriminative features
fraud = df[df['Class']==1].drop('Class',axis=1).mean()
legit = df[df['Class']==0].drop('Class',axis=1).mean()
diff  = (fraud - legit).abs().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,4))
diff.plot(kind='bar', color='#FF7043')
plt.title('Top 10 features: mean absolute difference (fraud vs legit)')
plt.ylabel('Absolute mean difference')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/feature_diff.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top discriminative features:', diff.index[:5].tolist())

---
## Phase 3 — Preprocessing & SMOTE

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Scale Amount and Time only (V1-V28 already PCA scaled)
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

features = [c for c in df.columns if c not in ['Class','Amount','Time']]
X = df[features]
y = df['Class']

# Stratified split — preserves 0.17% fraud ratio in each split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Train fraud rate: {y_train.mean()*100:.4f}%')
print(f'Val   fraud rate: {y_val.mean()*100:.4f}%')
print(f'Test  fraud rate: {y_test.mean()*100:.4f}%')

In [ ]:
# SMOTE — only on training data, NEVER on val/test
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print('Before SMOTE:', dict(y_train.value_counts()))
print('After  SMOTE:', dict(pd.Series(y_train_sm).value_counts()))
print('\nSMOTE creates synthetic fraud samples by interpolating between existing fraud points')

---
## Phase 4 — Multi-model training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                                   n_jobs=-1, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.1,
                                         random_state=42, eval_metric='auc'),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_sm, y_train_sm)
    probs = model.predict_proba(X_val)[:,1]
    auc   = roc_auc_score(y_val, probs)
    results[name] = {'model': model, 'probs': probs, 'auc': auc}
    print(f'  Val ROC-AUC: {auc:.4f}')

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8,6))
colors = ['#2196F3','#4CAF50','#F44336']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_val, res['probs'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.4f})", color=color)
plt.plot([0,1],[0,1],'k--',alpha=0.4)
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curves — validation set')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 5 — Threshold tuning
Default 0.5 threshold is wrong for fraud. We maximise **F2-score** (recall weighted 2x).

In [ ]:
from sklearn.metrics import fbeta_score

best_name  = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
print(f'Best model: {best_name}')

probs      = best_model.predict_proba(X_val)[:,1]
thresholds = np.arange(0.1, 0.9, 0.01)
f2_scores  = []

for t in thresholds:
    preds = (probs >= t).astype(int)
    f2_scores.append(fbeta_score(y_val, preds, beta=2, zero_division=0))

best_t  = thresholds[np.argmax(f2_scores)]
best_f2 = max(f2_scores)
print(f'Best threshold: {best_t:.2f}  |  F2-score: {best_f2:.4f}')

plt.figure(figsize=(8,4))
plt.plot(thresholds, f2_scores, color='#FF7043')
plt.axvline(best_t, color='red', linestyle='--', label=f'Best = {best_t:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F2-score')
plt.title('Threshold tuning')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 6 — Error analysis

In [ ]:
preds   = (probs >= best_t).astype(int)
y_v_arr = np.array(y_val)

fn_mask = (y_v_arr == 1) & (preds == 0)
fp_mask = (y_v_arr == 0) & (preds == 1)
tp_mask = (y_v_arr == 1) & (preds == 1)

print(f'Missed fraud   (FN): {fn_mask.sum()}')
print(f'Falsely blocked(FP): {fp_mask.sum()}')
print(f'Fraud caught   (TP): {tp_mask.sum()}')
recall    = tp_mask.sum()/(tp_mask.sum()+fn_mask.sum())
print(f'\nRecall (fraud caught): {recall:.2%}')

In [ ]:
# Feature importance
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns)
    top10 = imp.sort_values(ascending=True).tail(10)
    top10.plot(kind='barh', figsize=(9,5), color='#2196F3')
    plt.title('Top 10 feature importances')
    plt.tight_layout()
    plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Top 5 features:', imp.sort_values(ascending=False).index[:5].tolist())

---
## Phase 7 — Final evaluation on TEST set
> Run this only ONCE — after all tuning is done on validation set.

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score)

test_probs = best_model.predict_proba(X_test)[:,1]
test_preds = (test_probs >= best_t).astype(int)

test_auc = roc_auc_score(y_test, test_probs)
test_f2  = fbeta_score(y_test, test_preds, beta=2, zero_division=0)
cm       = confusion_matrix(y_test, test_preds)

print(f'TEST ROC-AUC : {test_auc:.4f}  (target >= 0.95)')
print(f'TEST F2-score: {test_f2:.4f}')
print(f'\nConfusion Matrix:')
print(cm)
print('\n', classification_report(y_test, test_preds,
      target_names=['Legitimate','Fraud']))

In [ ]:
# Confusion matrix plot
plt.figure(figsize=(6,5))
labels = [['TN','FP'],['FN','TP']]
plt.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{labels[i][j]}\n{cm[i,j]:,}",
                 ha='center', va='center', fontsize=12,
                 color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.xticks([0,1],['Predicted Legit','Predicted Fraud'])
plt.yticks([0,1],['Actual Legit','Actual Fraud'])
plt.title(f'Confusion matrix (threshold={best_t:.2f})')
plt.colorbar()
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 8 — Save model & summary

In [ ]:
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler,     '../models/scaler.pkl')
joblib.dump(best_t,     '../models/best_threshold.pkl')
print('Model saved to models/')
print(f'\n=== FINAL SUMMARY ===')
print(f'Best model    : {best_name}')
print(f'Test ROC-AUC  : {test_auc:.4f}  {"PASSED" if test_auc >= 0.95 else "BELOW TARGET"}')
print(f'Test F2-score : {test_f2:.4f}')
print(f'Threshold     : {best_t:.2f}')